# Task 3: A/B Hypothesis Testing

**Objective:** validate key risk-driver hypotheses for ACIS using statistically defensible A/B comparisons.

**Risk KPIs used in this notebook**
- **Claim Frequency:** proportion of policies with at least one claim.
- **Claim Severity:** average claim amount, conditional on a claim occurring.
- **Margin:** `TotalPremium - TotalClaims`.

**Null hypotheses tested**
1. No risk differences across provinces.
2. No risk differences between zip-code clusters.
3. No significant margin difference between zip-code clusters.
4. No risk difference between women and men.

**Decision rule:** reject H₀ when `p < 0.05`; otherwise fail to reject.

**Segment choice for the zip-code hypotheses**
- The sample contains two populated zip clusters, `2000-series` and `8000-series`.
- These clusters are used as the A/B comparison because they are the only sufficiently populated zip groups in the sample.
- This keeps the analysis reproducible while preserving a comparable control/test structure.


## Setup and Data Loading

Load the prepared insurance dataset and import the reusable hypothesis-testing helpers from `src`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_loader import load_and_prepare
from hypothesis_tests import run_task3_analysis


df = load_and_prepare(str(ROOT / "data" / "insurance_data.csv"))
print(f"Dataset loaded: {df.shape}")
df.head()

## Run Hypothesis Tests

The next cell runs the four Task 3 hypotheses and returns a results table with the selected KPI, test statistic, p-value, and decision.

In [ ]:
results = run_task3_analysis(df, alpha=0.05)

results_table = results[
    [
        "hypothesis",
        "feature",
        "group_a",
        "group_b",
        "kpi",
        "test",
        "statistic",
        "p_value",
        "decision",
    ]
].copy()

print("HYPOTHESIS TEST RESULTS")
print("=" * 120)
display(results_table)
print("\nDecision rule: reject H0 when p < 0.05")

## Business Recommendations

The following cell turns each statistical result into a short business-facing recommendation.

In [ ]:
print("BUSINESS RECOMMENDATIONS")
print("=" * 120)

for _, row in results.iterrows():
    print(f"\nHypothesis: {row['hypothesis']}")
    print(f"Group A: {row['group_a']} | Group B: {row['group_b']}")
    print(f"KPI: {row['kpi']} | Test: {row['test']} | p-value: {row['p_value']:.6f}")
    print(f"Decision: {row['decision']}")
    print(f"Recommendation: {row['recommendation']}")

print("\n" + "=" * 120)

## Export Results

Persist the results table to `reports/hypothesis_testing_results.csv` for submission and reuse.

In [ ]:
report_path = ROOT / "reports" / "hypothesis_testing_results.csv"
results.to_csv(report_path, index=False)
print(f"Saved results to: {report_path}")